<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/GraphicalNeuralNetworks/blob/main/FraudDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch-geometric -q

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.data import Data
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base = "/content/drive/MyDrive/fddataset/elliptic_bitcoin_dataset"
features = pd.read_csv(f"{base}/elliptic_txs_features.csv", header=None)
classes = pd.read_csv(f"{base}/elliptic_txs_classes.csv")
edges = pd.read_csv(f"{base}/elliptic_txs_edgelist.csv")

In [ ]:
features[:15]


In [ ]:
features.head()
classes.head()
edges.head()

In [ ]:
print(features.shape)
print(classes.shape)
print(edges.shape)

In [ ]:
print(classes["class"].value_counts())

In [ ]:
tx_ids=features.iloc[:, 0].values
tx_to_idx={
    tx_id:idx
    for idx,tx_id in enumerate(tx_ids)
}

In [ ]:
x=torch.tensor(features.iloc[:, 1:].values,dtype=torch.float)
print(x.shape)

In [ ]:
src=edges.iloc[:, 0].map(tx_to_idx)
dst=edges.iloc[:, 1].map(tx_to_idx)

edge_index = torch.tensor([src.values, dst.values],dtype=torch.long)
print(edge_index.shape)

In [ ]:
label_map={
    "1": 1,
    "2": 0,
    "unknown": -1
}

In [ ]:
labels=classes.copy()
labels["class"]=labels["class"].astype(str)
labels["class"]=labels["class"].map(label_map)

In [ ]:
labels=labels.set_index("txId")
labels=labels.reindex(tx_ids)
y=torch.tensor(labels["class"].values,dtype=torch.long)

In [ ]:
known_mask=y!=-1

In [ ]:
data = Data(
    x=x,
    edge_index=edge_index,
    y=y
)
data.known_mask=known_mask

In [ ]:
print(data)

In [ ]:
sample_nodes=set(features.iloc[:200, 0])
sample_edges=edges[ edges.iloc[:,0].isin(sample_nodes) & edges.iloc[:,1].isin(sample_nodes)]

G=nx.from_pandas_edgelist(
    sample_edges,
    source=sample_edges.columns[0],
    target=sample_edges.columns[1],
    create_using=nx.DiGraph()
)

plt.figure(figsize=(10,10))

nx.draw_networkx(
    G,
    node_size=40,
    arrows=False,
    with_labels=False
)

plt.show()

In [ ]:
class FraudDetector(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1=SAGEConv(in_channels, 64)
        self.conv2=SAGEConv(64, 64)
        self.fc=nn.Linear(64, 2)

    def forward(self, data):
        x=self.conv1(data.x, data.edge_index)
        x=F.relu(x)
        x=F.dropout(x,p=0.5,training=self.training)
        x=self.conv2(x,data.edge_index)
        x=F.relu(x)
        x=self.fc(x)
        return x

In [ ]:
model=FraudDetector(data.num_node_features)
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)
criterion=nn.CrossEntropyLoss()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
known_indices=(data.y!=-1).nonzero(as_tuple=True)[0]

In [ ]:
known_labels=data.y[known_indices]

In [ ]:
train_idx,test_idx=train_test_split(
    known_indices,
    test_size=0.15,
    random_state=42,
    stratify=known_labels
)

In [ ]:
train_labels=data.y[train_idx]

train_idx,val_idx=train_test_split(
    train_idx,
    test_size=0.1765,
    random_state=42,
    stratify=train_labels
)

In [ ]:
data.train_mask=torch.zeros(data.num_nodes, dtype=torch.bool)
data.val_mask=torch.zeros(data.num_nodes, dtype=torch.bool)
data.test_mask=torch.zeros(data.num_nodes, dtype=torch.bool)

data.train_mask[train_idx]=True
data.val_mask[val_idx]=True
data.test_mask[test_idx]=True

In [ ]:
epochs=100
for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()
    out=model(data)
    loss=criterion(out[data.train_mask],data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        out=model(data)
        pred=out.argmax(dim=1)
        val_correct=(pred[data.val_mask]==data.y[data.val_mask]).sum()
        val_acc=val_correct/data.val_mask.sum()

    print(f"Epoch {epoch+1:03d} Loss: {loss:.4f} Val Acc: {val_acc:.4f}")

In [ ]:
unknown_mask=data.y==-1
model.eval()
with torch.no_grad():
    out=model(data)
    pred=out.argmax(dim=1)
    test_correct=(pred[data.test_mask]==data.y[data.test_mask]).sum()
    test_acc=test_correct/data.test_mask.sum()
    pred_unknown = pred[unknown_mask]
print(f"Test Accuracy:{test_acc:.4f}")
print(pred_unknown[:1000])

In [ ]:
unknown_mask=data.y==-1
unknown_pred=pred[unknown_mask]
print(torch.unique(unknown_pred,return_counts=True))